# Occupancy Model Training & Comparison

Train and compare per-bay ROI classifiers for parking occupancy detection.

**Pipeline**: Full image → ROI crop (128x128 via `grid_sample`) → backbone → 2-class logits (free/occupied)

**Datasets**: PKLot (~712K patches) + CNRPark-EXT (~145K patches)

**Backbones to compare**: ResNet50 (baseline), ResNet101, EfficientNet-B3, ConvNeXt-Small

**Environment**: Works on both local (MPS/CPU) and Google Colab (T4 GPU)

## 1. Setup & Environment Detection

In [ ]:
import sys, os, subprocess, shutil
from pathlib import Path

# --- Detect environment ---
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    # Install dependencies
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "torchvision", "scikit-learn", "matplotlib"])
    # Mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

    DATA_DIR = Path("/content/data")
    DATA_DIR.mkdir(exist_ok=True)

    # Extract datasets from Drive (uploaded as tar.gz)
    DRIVE_DIR = Path("/content/drive/MyDrive/hack26_data")
    for archive_name, marker in [("pklot.tar.gz", "PKLot"), ("cnrpark.tar.gz", "CNRPark")]:
        archive = DRIVE_DIR / archive_name
        if archive.exists() and not (DATA_DIR / marker).exists():
            print(f"Extracting {archive_name}...")
            subprocess.check_call(["tar", "xzf", str(archive), "-C", str(DATA_DIR)])
            print(f"  Done: {archive_name}")
        elif not archive.exists():
            print(f"WARNING: {archive} not found on Drive")

    WEIGHTS_DIR = Path("/content/weights")
    WEIGHTS_DIR.mkdir(exist_ok=True)
else:
    PROJECT_ROOT = Path.cwd().parent
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    DATA_DIR = PROJECT_ROOT / "data"
    WEIGHTS_DIR = PROJECT_ROOT / "backend" / "model"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from torchvision.models import (
    resnet50, ResNet50_Weights,
    resnet101, ResNet101_Weights,
    efficientnet_b3, EfficientNet_B3_Weights,
    convnext_small, ConvNeXt_Small_Weights,
)
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
import time
import json
import copy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = 2 if IN_COLAB else 0
PIN_MEMORY = DEVICE.type == "cuda"

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Device: {DEVICE} | AMP: {USE_AMP} | Workers: {NUM_WORKERS}")
print(f"PyTorch: {torch.__version__}")
print(f"Data dir: {DATA_DIR}")

# Training config
IMG_SIZE = 128
BATCH_SIZE = 64
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Load Datasets

### PKLot
12,416 full parking lot images with polygon annotations (~712K parking spaces). Labels in FiftyOne `samples.json` format. We extract per-space bounding boxes and crop patches.

### CNRPark-EXT
~145K pre-cropped 150x150 patches from 9 cameras. Labels in `LABELS/all.txt`.

On Colab, datasets are extracted from `Google Drive/hack26_data/*.tar.gz`. Locally, they're in `data/`.

In [ ]:
# --- PKLot: Parse samples.json to extract (image_path, bbox, label) tuples ---
print("Parsing PKLot samples.json...")

with open(DATA_DIR / "PKLot" / "samples.json") as f:
    pklot_raw = json.load(f)

pklot_records = []  # (image_path, [x1,y1,x2,y2] normalized, label)
skipped = 0

for sample in pklot_raw["samples"]:
    img_path = DATA_DIR / "PKLot" / sample["filepath"]
    for space in sample["parking_spaces"]["polylines"]:
        # Extract bounding box from polygon points (normalized 0-1)
        points = space["points"]
        if not points or not points[0]:
            skipped += 1
            continue

        coords = points[0]  # list of [x, y] pairs
        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
        bbox = [min(xs), min(ys), max(xs), max(ys)]

        label = 1 if space["occupancy_status"] == "occupied" else 0
        pklot_records.append((str(img_path), bbox, label))

pklot_occupied = sum(r[2] for r in pklot_records)
print(f"PKLot: {len(pklot_records)} patches from {len(pklot_raw['samples'])} images (skipped {skipped} with empty polygons)")
print(f"  Occupied: {pklot_occupied} ({pklot_occupied/len(pklot_records)*100:.1f}%)")
print(f"  Empty:    {len(pklot_records)-pklot_occupied} ({(len(pklot_records)-pklot_occupied)/len(pklot_records)*100:.1f}%)")

In [ ]:
# --- CNRPark-EXT: Parse LABELS/all.txt for (patch_path, label) tuples ---
cnrpark_dir = DATA_DIR / "CNRPark" / "CNR-EXT-Patches-150x150"
labels_file = cnrpark_dir / "LABELS" / "all.txt"

if labels_file.exists():
    cnrpark_records = []  # (patch_path, label)
    with open(labels_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.rsplit(" ", 1)
            rel_path, label = parts[0], int(parts[1])
            patch_path = cnrpark_dir / "PATCHES" / rel_path
            if patch_path.exists():
                cnrpark_records.append((str(patch_path), label))

    cnrpark_occupied = sum(r[1] for r in cnrpark_records)
    print(f"CNRPark-EXT: {len(cnrpark_records)} patches")
    print(f"  Occupied: {cnrpark_occupied} ({cnrpark_occupied/len(cnrpark_records)*100:.1f}%)")
    print(f"  Empty:    {len(cnrpark_records)-cnrpark_occupied} ({(len(cnrpark_records)-cnrpark_occupied)/len(cnrpark_records)*100:.1f}%)")
    HAS_CNRPARK = True
else:
    print(f"CNRPark labels not found at {labels_file}")
    cnrpark_records = []
    HAS_CNRPARK = False

## 3. Data Exploration

Visualize samples from both classes and both datasets to understand what the model will see.

In [ ]:
def show_pklot_samples(records, title, n=8):
    """Show n occupied and n empty PKLot patches (cropped from full images)."""
    occupied = [r for r in records if r[2] == 1][:n]
    empty = [r for r in records if r[2] == 0][:n]

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for col, (img_path, bbox, _) in enumerate(occupied):
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        crop = img.crop((int(bbox[0]*w), int(bbox[1]*h), int(bbox[2]*w), int(bbox[3]*h)))
        axes[0, col].imshow(crop)
        axes[0, col].set_title("Occupied", fontsize=9, color="red")
        axes[0, col].axis("off")

    for col, (img_path, bbox, _) in enumerate(empty):
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        crop = img.crop((int(bbox[0]*w), int(bbox[1]*h), int(bbox[2]*w), int(bbox[3]*h)))
        axes[1, col].imshow(crop)
        axes[1, col].set_title("Empty", fontsize=9, color="green")
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.show()


def show_cnrpark_samples(records, title, n=8):
    """Show n occupied and n empty CNRPark patches."""
    occupied = [r for r in records if r[1] == 1][:n]
    empty = [r for r in records if r[1] == 0][:n]

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for col, (path, _) in enumerate(occupied):
        img = Image.open(path).convert("RGB")
        axes[0, col].imshow(img)
        axes[0, col].set_title("Occupied", fontsize=9, color="red")
        axes[0, col].axis("off")

    for col, (path, _) in enumerate(empty):
        img = Image.open(path).convert("RGB")
        axes[1, col].imshow(img)
        axes[1, col].set_title("Empty", fontsize=9, color="green")
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.show()

# Shuffle for variety before showing
import random
rng = random.Random(SEED)
pklot_shuffled = pklot_records.copy()
rng.shuffle(pklot_shuffled)
show_pklot_samples(pklot_shuffled, "PKLot Samples (cropped from full images)")

if HAS_CNRPARK:
    cnrpark_shuffled = cnrpark_records.copy()
    rng.shuffle(cnrpark_shuffled)
    show_cnrpark_samples(cnrpark_shuffled, "CNRPark-EXT Samples (pre-cropped patches)")

## 4. Dataset & Augmentation Pipeline

Wrap local data into PyTorch Datasets with augmentations for training.

In [ ]:
class PKLotPatchDataset(Dataset):
    """Loads PKLot patches by cropping bounding boxes from full images on-the-fly."""

    def __init__(self, records, transform=None):
        """records: list of (image_path, [x1,y1,x2,y2] normalized, label)"""
        self.records = records
        self.transform = transform
        # Cache: group records by image path to reduce redundant I/O
        self._img_cache_path = None
        self._img_cache = None

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        img_path, bbox, label = self.records[idx]

        # Simple single-image cache (helps when consecutive patches share an image)
        if self._img_cache_path != img_path:
            self._img_cache = Image.open(img_path).convert("RGB")
            self._img_cache_path = img_path

        img = self._img_cache
        w, h = img.size
        crop = img.crop((
            int(bbox[0] * w), int(bbox[1] * h),
            int(bbox[2] * w), int(bbox[3] * h),
        ))

        if self.transform:
            crop = self.transform(crop)
        return crop, label


class CNRParkPatchDataset(Dataset):
    """Loads pre-cropped CNRPark-EXT patches."""

    def __init__(self, records, transform=None):
        """records: list of (patch_path, label)"""
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, label = self.records[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


# ImageNet normalization (matching our predictor.py preprocessing)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f"Train augmentations: {train_transform}")
print(f"Val augmentations:   {val_transform}")

In [ ]:
# Split PKLot records: 80% train, 10% val, 10% test
# Shuffle deterministically, then split by index
import random
rng_split = random.Random(SEED)
pklot_shuffled_all = pklot_records.copy()
rng_split.shuffle(pklot_shuffled_all)

n_total = len(pklot_shuffled_all)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)

pklot_train_records = pklot_shuffled_all[:n_train]
pklot_val_records = pklot_shuffled_all[n_train:n_train + n_val]
pklot_test_records = pklot_shuffled_all[n_train + n_val:]

# Build datasets
pklot_train = PKLotPatchDataset(pklot_train_records, transform=train_transform)
pklot_val = PKLotPatchDataset(pklot_val_records, transform=val_transform)
pklot_test = PKLotPatchDataset(pklot_test_records, transform=val_transform)

print(f"PKLot splits: train={len(pklot_train)}, val={len(pklot_val)}, test={len(pklot_test)}")

# CNRPark-EXT for cross-dataset evaluation
if HAS_CNRPARK:
    cnrpark_eval = CNRParkPatchDataset(cnrpark_records, transform=val_transform)
    print(f"CNRPark-EXT: {len(cnrpark_eval)} patches for cross-dataset eval")

# DataLoaders (NUM_WORKERS / PIN_MEMORY set in cell 1 based on environment)
train_loader = DataLoader(pklot_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(pklot_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(pklot_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

if HAS_CNRPARK:
    cross_loader = DataLoader(cnrpark_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# Verify a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}, labels: {labels[:10]}")

## 5. Model Definitions

Each backbone takes a 128x128 RGB crop and outputs 2-class logits (free/occupied).
All use ImageNet pretrained weights with the last few layers unfrozen for fine-tuning.

In [ ]:
def build_resnet50():
    """ResNet50 — our current production baseline."""
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(2048, 2)
    # Freeze early layers, train layer2+
    for name, param in model.named_parameters():
        if not any(name.startswith(layer) for layer in ["layer2", "layer3", "layer4", "fc"]):
            param.requires_grad_(False)
    return model


def build_resnet101():
    """ResNet101 — deeper, should capture finer features."""
    model = resnet101(weights=ResNet101_Weights.DEFAULT)
    model.fc = nn.Linear(2048, 2)
    for name, param in model.named_parameters():
        if not any(name.startswith(layer) for layer in ["layer2", "layer3", "layer4", "fc"]):
            param.requires_grad_(False)
    return model


def build_efficientnet_b3():
    """EfficientNet-B3 — compound-scaled architecture, strong accuracy/efficiency."""
    model = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    # Freeze features up to block 5 (of 7)
    for i, block in enumerate(model.features):
        if i < 5:
            for param in block.parameters():
                param.requires_grad_(False)
    return model


def build_convnext_small():
    """ConvNeXt-Small — modern architecture, top accuracy on ImageNet."""
    model = convnext_small(weights=ConvNeXt_Small_Weights.DEFAULT)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    # Freeze first 2 stages (of 4)
    for i, stage in enumerate(model.features):
        if i < 4:  # stages 0-3 (first 2 ConvNeXt stages = 4 Sequential blocks)
            for param in stage.parameters():
                param.requires_grad_(False)
    return model


MODELS = {
    "ResNet50": build_resnet50,
    "ResNet101": build_resnet101,
    "EfficientNet-B3": build_efficientnet_b3,
    "ConvNeXt-Small": build_convnext_small,
}

# Show parameter counts
for name, builder in MODELS.items():
    model = builder()
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name:20s}: {total/1e6:.1f}M params, {trainable/1e6:.1f}M trainable")

## 6. Training Loop

Train each backbone with:
- **Optimizer**: AdamW (lr=1e-4, weight_decay=1e-4)
- **Scheduler**: CosineAnnealingLR over 15 epochs
- **Early stopping**: patience=4 on val loss
- **Mixed precision**: via `torch.amp` for faster training on GPU/MPS

In [ ]:
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4

# Use AMP only on CUDA (MPS support is limited)
USE_AMP = DEVICE.type == "cuda"


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    """Train for one epoch, return average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        if USE_AMP:
            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate model, return loss, accuracy, and all predictions/labels."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    total = len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    return running_loss / total, acc, np.array(all_preds), np.array(all_labels)


print("Training functions defined.")

In [ ]:
def train_model(model_name, builder_fn, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    """Full training loop for one model with early stopping."""
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")

    model = builder_fn().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(num_epochs):
        t0 = time.time()

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        elapsed = time.time() - t0
        lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"  Epoch {epoch+1:2d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
            f"lr={lr:.2e} | {elapsed:.0f}s"
        )

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1} (patience={PATIENCE})")
                break

    # Restore best weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, history


# Train all models
results = {}
trained_models = {}

for model_name, builder_fn in MODELS.items():
    model, history = train_model(model_name, builder_fn, train_loader, val_loader)
    trained_models[model_name] = model
    results[model_name] = history

print(f"\n{'='*60}")
print("All models trained!")
print(f"{'='*60}")

## 7. Training Curves

Plot loss and accuracy curves for all backbones side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for i, (name, hist) in enumerate(results.items()):
    epochs = range(1, len(hist["train_loss"]) + 1)
    axes[0].plot(epochs, hist["train_loss"], "--", color=colors[i], alpha=0.5)
    axes[0].plot(epochs, hist["val_loss"], "-", color=colors[i], label=name)

    axes[1].plot(epochs, hist["train_acc"], "--", color=colors[i], alpha=0.5)
    axes[1].plot(epochs, hist["val_acc"], "-", color=colors[i], label=name)

axes[0].set_title("Loss (dashed=train, solid=val)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Accuracy (dashed=train, solid=val)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(DATA_DIR / "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {DATA_DIR / 'training_curves.png'}")

## 8. Evaluation on PKLot Test Set

Evaluate all trained models on the held-out PKLot test split. Compare accuracy, precision, recall, F1 and show confusion matrices.

In [ ]:
criterion = nn.CrossEntropyLoss()
test_results = {}

print("Evaluating on PKLot test set...\n")
for name, model in trained_models.items():
    t0 = time.time()
    test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion)
    elapsed = time.time() - t0

    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    test_results[name] = {
        "accuracy": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loss": test_loss,
        "preds": preds,
        "labels": labels,
        "eval_time": elapsed,
    }

    print(f"{name:20s} | acc={test_acc:.4f} | prec={prec:.4f} | rec={rec:.4f} | f1={f1:.4f} | loss={test_loss:.4f} | {elapsed:.1f}s")

# Summary table
print(f"\n{'Model':20s} {'Accuracy':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s}")
print("-" * 62)
for name, r in sorted(test_results.items(), key=lambda x: -x[1]["f1"]):
    print(f"{name:20s} {r['accuracy']:10.4f} {r['precision']:10.4f} {r['recall']:10.4f} {r['f1']:10.4f}")

best_model_name = max(test_results, key=lambda k: test_results[k]["f1"])
print(f"\nBest model by F1: {best_model_name} (F1={test_results[best_model_name]['f1']:.4f})")

In [ ]:
# Confusion matrices
n_models = len(test_results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

class_names = ["Empty", "Occupied"]

for ax, (name, r) in zip(axes, test_results.items()):
    cm = confusion_matrix(r["labels"], r["preds"])
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"{name}\nAcc={r['accuracy']:.4f} F1={r['f1']:.4f}")

plt.tight_layout()
plt.savefig(str(DATA_DIR / "confusion_matrices_pklot.png"), dpi=150, bbox_inches="tight")
plt.show()

## 9. Cross-Dataset Validation (CNRPark-EXT)

The real test of generalization: models trained on PKLot, evaluated on CNRPark-EXT (different country, cameras, angles, lighting). This simulates deploying to a new parking lot without any fine-tuning.

In [ ]:
if not HAS_CNRPARK:
    print("CNRPark-EXT not available — skipping cross-dataset validation.")
    cross_results = {}
else:
    cross_results = {}

    print("Cross-dataset evaluation: trained on PKLot, evaluated on CNRPark-EXT\n")
    for name, model in trained_models.items():
        t0 = time.time()
        cross_loss, cross_acc, preds, labels = evaluate(model, cross_loader, criterion)
        elapsed = time.time() - t0

        prec = precision_score(labels, preds, zero_division=0)
        rec = recall_score(labels, preds, zero_division=0)
        f1 = f1_score(labels, preds, zero_division=0)

        cross_results[name] = {
            "accuracy": cross_acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "loss": cross_loss,
            "preds": preds,
            "labels": labels,
        }

        print(f"{name:20s} | acc={cross_acc:.4f} | prec={prec:.4f} | rec={rec:.4f} | f1={f1:.4f} | {elapsed:.1f}s")

    # Comparison table: PKLot test vs CNRPark cross-eval
    print(f"\n{'Model':20s} {'PKLot F1':>10s} {'CNRPark F1':>12s} {'Delta':>8s}")
    print("-" * 52)
    for name in test_results:
        pklot_f1 = test_results[name]["f1"]
        cnr_f1 = cross_results[name]["f1"]
        delta = cnr_f1 - pklot_f1
        print(f"{name:20s} {pklot_f1:10.4f} {cnr_f1:12.4f} {delta:+8.4f}")

    best_cross = max(cross_results, key=lambda k: cross_results[k]["f1"])
    print(f"\nBest cross-dataset generalization: {best_cross} (F1={cross_results[best_cross]['f1']:.4f})")

In [ ]:
# Cross-dataset confusion matrices
if cross_results:
    n_models = len(cross_results)
    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
    if n_models == 1:
        axes = [axes]

    for ax, (name, r) in zip(axes, cross_results.items()):
        cm = confusion_matrix(r["labels"], r["preds"])
        disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
        disp.plot(ax=ax, cmap="Oranges", values_format="d")
        ax.set_title(f"{name} on CNRPark\nAcc={r['accuracy']:.4f} F1={r['f1']:.4f}")

    plt.suptitle("Cross-Dataset: Trained on PKLot, Evaluated on CNRPark-EXT", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(DATA_DIR / "confusion_matrices_cnrpark.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 10. Error Analysis

Examine the hardest examples: misclassified patches with highest confidence. These reveal systematic failure modes (shadows, partial occlusion, edge cases).

In [ ]:
@torch.no_grad()
def get_confident_errors(model, loader, n=16):
    """Find the n misclassified samples with the highest model confidence."""
    model.eval()
    errors = []  # (confidence, pred, true_label, image_tensor)

    for images, labels in loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = outputs.softmax(1)
        preds = probs.argmax(1)

        for i in range(len(labels)):
            if preds[i].item() != labels[i].item():
                conf = probs[i, preds[i]].item()
                errors.append((conf, preds[i].item(), labels[i].item(), images[i].cpu()))

    errors.sort(key=lambda x: -x[0])
    return errors[:n]


# Show errors for the best model
best = trained_models[best_model_name]
errors = get_confident_errors(best, test_loader, n=16)

if errors:
    inv_normalize = transforms.Normalize(
        mean=[-m / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
        std=[1.0 / s for s in IMAGENET_STD],
    )

    n_show = min(16, len(errors))
    cols = min(8, n_show)
    rows = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 3 * rows))
    axes = np.array(axes).flatten()

    for i, (conf, pred, true, img_tensor) in enumerate(errors[:n_show]):
        img = inv_normalize(img_tensor).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[i].imshow(img)
        pred_label = "Occupied" if pred == 1 else "Empty"
        true_label = "Occupied" if true == 1 else "Empty"
        axes[i].set_title(f"P:{pred_label}\nT:{true_label}\n{conf:.2f}", fontsize=8)
        axes[i].axis("off")

    for i in range(n_show, len(axes)):
        axes[i].axis("off")

    plt.suptitle(f"Confident Errors — {best_model_name} on PKLot Test", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No errors found — perfect classification!")

## 11. Inference Speed Benchmark

Measure per-crop inference time on CPU and current device. This determines feasibility for real-time processing with 20+ bays per frame.

In [ ]:
@torch.no_grad()
def benchmark_model(model, device, batch_size=20, n_runs=50, warmup=5):
    """Measure inference time for a batch of `batch_size` crops (simulating 20 bays)."""
    model = model.to(device).eval()
    dummy = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=device)

    # Warmup
    for _ in range(warmup):
        _ = model(dummy)

    times = []
    for _ in range(n_runs):
        t0 = time.time()
        _ = model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elif device.type == "mps":
            torch.mps.synchronize()
        times.append((time.time() - t0) * 1000)

    return np.median(times), np.mean(times), np.std(times)


print(f"Benchmarking on {DEVICE} — batch of 20 crops (simulating 20 bays per frame)\n")
print(f"{'Model':20s} {'Median (ms)':>12s} {'Mean (ms)':>10s} {'Std (ms)':>10s} {'Per-crop (ms)':>14s}")
print("-" * 68)

speed_results = {}
for name, model in trained_models.items():
    median_ms, mean_ms, std_ms = benchmark_model(model, DEVICE)
    speed_results[name] = {"median_ms": median_ms, "mean_ms": mean_ms, "per_crop_ms": median_ms / 20}
    print(f"{name:20s} {median_ms:12.1f} {mean_ms:10.1f} {std_ms:10.1f} {median_ms/20:14.2f}")

# Also benchmark on CPU for production reference
print(f"\nCPU benchmark (production):")
print(f"{'Model':20s} {'Median (ms)':>12s} {'Per-crop (ms)':>14s}")
print("-" * 48)

cpu_device = torch.device("cpu")
for name, model in trained_models.items():
    median_ms, _, _ = benchmark_model(model, cpu_device, n_runs=20)
    speed_results[name]["cpu_median_ms"] = median_ms
    print(f"{name:20s} {median_ms:12.1f} {median_ms/20:14.2f}")

## 12. Export Best Model

Save the best model's weights in a format compatible with the existing `backend/predictor.py` pipeline. Also export to ONNX for potential production speedup.

In [ ]:
# Decide best model: prioritize accuracy (F1), then cross-dataset generalization
if cross_results:
    # Weighted score: 60% PKLot F1 + 40% CNRPark F1
    combined_scores = {
        name: 0.6 * test_results[name]["f1"] + 0.4 * cross_results[name]["f1"]
        for name in test_results
    }
    best_overall = max(combined_scores, key=combined_scores.get)
    print("Combined ranking (60% PKLot + 40% CNRPark F1):")
    for name, score in sorted(combined_scores.items(), key=lambda x: -x[1]):
        print(f"  {name:20s}: {score:.4f}  (PKLot={test_results[name]['f1']:.4f}, CNRPark={cross_results[name]['f1']:.4f})")
else:
    best_overall = best_model_name
    print(f"Best model (PKLot F1 only): {best_overall}")

print(f"\nSelected for export: {best_overall}")
best_model = trained_models[best_overall]

In [ ]:
# Save PyTorch weights
export_dir = WEIGHTS_DIR
export_dir.mkdir(parents=True, exist_ok=True)

# Save full model state dict for the best model
weights_path = export_dir / f"occupancy_{best_overall.lower().replace('-', '_')}_pklot.pt"
torch.save({
    "model_name": best_overall,
    "state_dict": best_model.state_dict(),
    "img_size": IMG_SIZE,
    "num_classes": 2,
    "class_names": ["empty", "occupied"],
    "metrics": {
        "pklot_test_f1": test_results[best_overall]["f1"],
        "pklot_test_acc": test_results[best_overall]["accuracy"],
        "cnrpark_f1": cross_results.get(best_overall, {}).get("f1"),
        "cnrpark_acc": cross_results.get(best_overall, {}).get("accuracy"),
    },
    "training_config": {
        "epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "dataset": "PKLot",
    },
}, weights_path)
print(f"Saved PyTorch weights to {weights_path}")
print(f"  Size: {weights_path.stat().st_size / 1024 / 1024:.1f} MB")

# Also save all models for later comparison
for name, model in trained_models.items():
    path = export_dir / f"occupancy_{name.lower().replace('-', '_')}_pklot.pt"
    torch.save({
        "model_name": name,
        "state_dict": model.state_dict(),
        "img_size": IMG_SIZE,
        "num_classes": 2,
    }, path)
    print(f"Saved {name} to {path.name} ({path.stat().st_size / 1024 / 1024:.1f} MB)")

# On Colab, copy weights back to Drive for download
if IN_COLAB:
    drive_weights = Path("/content/drive/MyDrive/hack26_data/weights")
    drive_weights.mkdir(parents=True, exist_ok=True)
    for pt_file in export_dir.glob("*.pt"):
        shutil.copy2(pt_file, drive_weights / pt_file.name)
    print(f"\nWeights copied to Google Drive: {drive_weights}")

In [ ]:
# Export to ONNX for production (1.5-3x CPU speedup via ONNX Runtime)
try:
    import onnx

    best_model_cpu = copy.deepcopy(best_model).cpu().eval()
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    onnx_path = export_dir / f"occupancy_{best_overall.lower().replace('-', '_')}_pklot.onnx"

    torch.onnx.export(
        best_model_cpu,
        dummy_input,
        str(onnx_path),
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        opset_version=17,
    )

    # Validate
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print(f"ONNX exported to {onnx_path}")
    print(f"  Size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB")

    # Benchmark ONNX if onnxruntime is available
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(str(onnx_path))
        dummy_np = dummy_input.numpy()

        # Warmup
        for _ in range(5):
            sess.run(None, {"input": np.repeat(dummy_np, 20, axis=0)})

        times = []
        batch_np = np.repeat(dummy_np, 20, axis=0)  # 20 bays
        for _ in range(50):
            t0 = time.time()
            sess.run(None, {"input": batch_np})
            times.append((time.time() - t0) * 1000)

        median_ms = np.median(times)
        print(f"  ONNX Runtime (CPU, 20 bays): {median_ms:.1f}ms ({median_ms/20:.2f}ms/crop)")

        # Compare with PyTorch CPU
        if "cpu_median_ms" in speed_results.get(best_overall, {}):
            pt_ms = speed_results[best_overall]["cpu_median_ms"]
            speedup = pt_ms / median_ms
            print(f"  Speedup vs PyTorch CPU: {speedup:.1f}x")
    except ImportError:
        print("  onnxruntime not installed — install with: pip install onnxruntime")

except ImportError:
    print("onnx not installed — skipping ONNX export. Install with: pip install onnx onnxruntime")

## 13. Final Summary

Comprehensive comparison of all models across all evaluation dimensions.

In [ ]:
# Final summary table
print("=" * 90)
print("FINAL MODEL COMPARISON")
print("=" * 90)

header = (
    f"{'Model':20s} {'PKLot F1':>9s} {'PKLot Acc':>10s}"
    + (f" {'CNR F1':>8s} {'CNR Acc':>9s}" if cross_results else "")
    + f" {'CPU 20bay':>10s} {'Params':>8s}"
)
print(header)
print("-" * len(header))

for name in MODELS:
    tr = test_results[name]
    sr = speed_results.get(name, {})
    model = trained_models[name]
    n_params = sum(p.numel() for p in model.parameters()) / 1e6

    row = f"{name:20s} {tr['f1']:9.4f} {tr['accuracy']:10.4f}"
    if cross_results:
        cr = cross_results[name]
        row += f" {cr['f1']:8.4f} {cr['accuracy']:9.4f}"
    cpu_ms = sr.get("cpu_median_ms", 0)
    row += f" {cpu_ms:8.1f}ms {n_params:7.1f}M"
    print(row)

print("-" * len(header))
print(f"\nSelected model: {best_overall}")
print(f"Weights saved to: {WEIGHTS_DIR}")

# Save results JSON for reference
results_summary = {
    "selected_model": best_overall,
    "models": {},
}
for name in MODELS:
    results_summary["models"][name] = {
        "pklot_test": {
            "accuracy": round(test_results[name]["accuracy"], 4),
            "precision": round(test_results[name]["precision"], 4),
            "recall": round(test_results[name]["recall"], 4),
            "f1": round(test_results[name]["f1"], 4),
        },
        "speed": {
            "cpu_20bay_ms": round(speed_results.get(name, {}).get("cpu_median_ms", 0), 1),
        },
    }
    if cross_results and name in cross_results:
        results_summary["models"][name]["cnrpark_cross"] = {
            "accuracy": round(cross_results[name]["accuracy"], 4),
            "f1": round(cross_results[name]["f1"], 4),
        }

results_path = DATA_DIR / "model_comparison_results.json"
with open(results_path, "w") as f:
    json.dump(results_summary, f, indent=2)
print(f"\nResults saved to {results_path}")